# 18. Scent Term Dictionary Validation

17번 IFRA Coverage 분석에서 확인된 P0 고빈도 unmatched / alias 후보 8개를 **IFRA, Givaudan, IFF 공식 자료만으로** 검증하고, 재사용 가능한 향 용어 매핑 사전 v0.1을 만든다.

이번 노트북은 임베딩, fuzzy similarity, 일반 웹 검색, LLM 자체 지식을 Evidence로 사용하지 않는다. 관계 판정이 먼저이며 Coverage 변화는 판정 결과의 영향만 평가한다.


## 0. 핵심 질문과 판정 범위

P0 대상: `Woody Notes`, `Green Notes`, `Floral Notes`, `Citruses`, `Black Currant`, `White Musk`, `Oud`, `Cedar`.

판정 유형:

- `SAME_CONCEPT`: 공식 자료가 향 용어 수준의 동일 개념을 직접 또는 매우 명확하게 지지
- `FAMILY`: 더 넓은/좁은 향 profile 또는 복수 원료 family 관계
- `RELATED`: 관련은 있지만 동일 개념이나 family로 합치면 안 됨
- `UNRESOLVED`: 허용된 공식 근거만으로 확정 불가

Canonicalization과 Ingredient Mapping을 분리한다. 예를 들어 `Oud -> Agarwood`가 용어 수준에서 확인되어도 `Oud Note = 특정 Agarwood oil`로 해석하지 않는다.


## 1. Imports / Paths


In [1]:
import collections
import json
import pathlib

import numpy as np
import pandas as pd
import pdfplumber
from IPython.display import display

PROJECT_ROOT = pathlib.Path.cwd().resolve()
PDF_PATH = PROJECT_ROOT / "data" / "external" / "ifra" / "raw" / "ifra-fragrance-ingredient-glossary-april-2020.pdf"
IFRA_PATH = PROJECT_ROOT / "data" / "external" / "ifra" / "processed" / "ifra_ingredients_2020.csv"
DEFINITION_PATH = PROJECT_ROOT / "data" / "external" / "ifra" / "processed" / "ifra_primary_descriptor_definitions_2020.csv"
NOTE_MATCH_PATH = PROJECT_ROOT / "analysis_outputs" / "17_note_ifra_matching.csv"
ALIAS_PATH = PROJECT_ROOT / "analysis_outputs" / "17_ifra_alias_candidates.csv"
UNMATCHED_PATH = PROJECT_ROOT / "analysis_outputs" / "17_top_unmatched_notes.csv"
METRICS_PATH = PROJECT_ROOT / "analysis_outputs" / "17_ifra_coverage_metrics.csv"
STAGE17_SUMMARY_PATH = PROJECT_ROOT / "analysis_outputs" / "17_ifra_knowledge_coverage_summary.md"
JSONL_PATH = PROJECT_ROOT / "perfumes.jsonl"
CSV_PATH = PROJECT_ROOT / "perfumes.csv"
SCHEMA_PATH = PROJECT_ROOT / "SCHEMA.md"

KNOWLEDGE_DIR = PROJECT_ROOT / "data" / "scent_knowledge"
OUTPUT_DIR = PROJECT_ROOT / "analysis_outputs"
DICTIONARY_PATH = KNOWLEDGE_DIR / "scent_term_dictionary_v0.1.csv"
EVIDENCE_PATH = KNOWLEDGE_DIR / "scent_term_evidence_v0.1.csv"
DECISION_PATH = OUTPUT_DIR / "18_scent_term_validation_decisions.csv"
SUMMARY_PATH = OUTPUT_DIR / "18_scent_term_dictionary_validation_summary.md"

KNOWLEDGE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

P0_TERMS = ["Woody Notes", "Green Notes", "Floral Notes", "Citruses", "Black Currant", "White Musk", "Oud", "Cedar"]
RETRIEVED_AT = "2026-09-01"
VERSION = "v0.1"

def show(df, n=30):
    display(df.head(n).style.hide(axis="index"))

def ratio(numerator, denominator):
    return float(numerator / denominator) if denominator else 0.0


## 2. Validate Existing Project Files


In [2]:
required_inputs = [
    CSV_PATH, JSONL_PATH, SCHEMA_PATH, PDF_PATH, IFRA_PATH, DEFINITION_PATH,
    NOTE_MATCH_PATH, ALIAS_PATH, UNMATCHED_PATH, METRICS_PATH, STAGE17_SUMMARY_PATH,
]
input_check_df = pd.DataFrame([
    {
        "path": str(path.relative_to(PROJECT_ROOT)),
        "exists": path.exists(),
        "size_mb": round(path.stat().st_size / 1024**2, 3) if path.exists() else np.nan,
    }
    for path in required_inputs
])
show(input_check_df)
assert input_check_df["exists"].all(), input_check_df.loc[~input_check_df["exists"], "path"].tolist()

with pdfplumber.open(PDF_PATH) as pdf:
    pdf_page_count = len(pdf.pages)
    definition_source_text = "\n".join((pdf.pages[index].extract_text() or "") for index in range(6, 12))
assert pdf_page_count == 111
for term in ["WOODY", "GREEN", "FLORAL", "CITRUS", "MUSK-LIKE"]:
    assert term in definition_source_text
print(f"Validated {len(required_inputs)} inputs; IFRA PDF pages={pdf_page_count}.")


path,exists,size_mb
perfumes.csv,True,121.429000
perfumes.jsonl,True,485.938000
SCHEMA.md,True,0.005000
data\external\ifra\raw\ifra-fragrance-ingredient-glossary-april-2020.pdf,True,1.122000
data\external\ifra\processed\ifra_ingredients_2020.csv,True,0.192000
data\external\ifra\processed\ifra_primary_descriptor_definitions_2020.csv,True,0.005000
analysis_outputs\17_note_ifra_matching.csv,True,0.115000
analysis_outputs\17_ifra_alias_candidates.csv,True,0.000000
analysis_outputs\17_top_unmatched_notes.csv,True,0.004000
analysis_outputs\17_ifra_coverage_metrics.csv,True,0.001000


Validated 11 inputs; IFRA PDF pages=111.


## 3. Stage 17 P0 Status


In [3]:
note_match_df = pd.read_csv(NOTE_MATCH_PATH, keep_default_na=False)
stage17_alias_df = pd.read_csv(ALIAS_PATH, keep_default_na=False)
stage17_unmatched_df = pd.read_csv(UNMATCHED_PATH, keep_default_na=False)
stage17_metrics_df = pd.read_csv(METRICS_PATH)
stage17_metrics = dict(zip(stage17_metrics_df["metric"], stage17_metrics_df["value"]))

p0_stage17_df = note_match_df[note_match_df["note"].isin(P0_TERMS)].copy()
p0_stage17_df["stage17_candidate"] = p0_stage17_df.apply(
    lambda row: " | ".join(value for value in [row["material_family_candidate"], row["alias_candidate"]] if value),
    axis=1,
)
p0_stage17_df = p0_stage17_df.rename(columns={"note": "term", "final_auto_status": "stage17_status"})[
    ["term", "perfume_count", "occurrence_count", "stage17_status", "stage17_candidate"]
]
p0_stage17_df["term"] = pd.Categorical(p0_stage17_df["term"], categories=P0_TERMS, ordered=True)
p0_stage17_df = p0_stage17_df.sort_values("term").reset_index(drop=True)
assert len(p0_stage17_df) == 8
show(p0_stage17_df, n=8)


term,perfume_count,occurrence_count,stage17_status,stage17_candidate
Woody Notes,8278,8385,UNMATCHED,
Green Notes,4529,4538,UNMATCHED,
Floral Notes,4436,4505,UNMATCHED,
Citruses,4982,4993,UNMATCHED,
Black Currant,5560,5566,UNMATCHED,
White Musk,8339,8371,UNMATCHED,
Oud,3780,3895,CANDIDATE_ONLY,Agarwood
Cedar,22305,22502,CANDIDATE_ONLY,"Cedar leaf oil | Cedar leaf oil, China | Cedar leaf oil, East Canada | Cedar leaf oil, rectified | Cedarwood"


## 4. Local IFRA Evidence


In [4]:
ifra_df = pd.read_csv(IFRA_PATH, keep_default_na=False)
definitions_df = pd.read_csv(DEFINITION_PATH, keep_default_na=False)

definition_terms = ["Woody", "Green", "Floral", "Citrus", "Musk-Like"]
selected_definitions_df = definitions_df[definitions_df["descriptor"].isin(definition_terms)].copy()

blackcurrant_mask = (
    ifra_df["descriptor_2"].str.casefold().eq("blackcurrant")
    | ifra_df["descriptor_3"].str.casefold().eq("blackcurrant")
)
cassis_rows_df = ifra_df[ifra_df["principal_name"].str.contains("Cassis", case=False, regex=False)].copy()
agarwood_rows_df = ifra_df[ifra_df["principal_name"].str.contains("Agarwood", case=False, regex=False)].copy()
cedarwood_rows_df = ifra_df[ifra_df["principal_name"].str.contains("Cedarwood", case=False, regex=False)].copy()
cedarwood_descriptor_mask = (
    ifra_df["descriptor_2"].str.casefold().eq("cedarwood")
    | ifra_df["descriptor_3"].str.casefold().eq("cedarwood")
)
musk_like_primary_mask = ifra_df["primary_descriptor"].str.casefold().str.replace("-", " ", regex=False).eq("musk like")

ifra_evidence_stats_df = pd.DataFrame([
    {"item": "Blackcurrant descriptor rows", "value": int(blackcurrant_mask.sum())},
    {"item": "Cassis principal-name rows", "value": len(cassis_rows_df)},
    {"item": "Agarwood principal-name rows", "value": len(agarwood_rows_df)},
    {"item": "Agarwood unique CAS", "value": agarwood_rows_df["cas_number"].nunique()},
    {"item": "Cedarwood principal-name rows", "value": len(cedarwood_rows_df)},
    {"item": "Cedarwood unique CAS", "value": cedarwood_rows_df["cas_number"].nunique()},
    {"item": "Cedarwood descriptor rows", "value": int(cedarwood_descriptor_mask.sum())},
    {"item": "Musk-like primary rows", "value": int(musk_like_primary_mask.sum())},
])
show(selected_definitions_df, n=10)
show(ifra_evidence_stats_df, n=20)


descriptor,definition,source_page
Citrus,"Citrus notes are given by the smell of fruit from the citrus family – such as orange, lemon or grapefruit.",8
Floral,"Floral notes belong to the large floral family that includes notes such as rose, jasmin, narcissus and others. Some fragrance materials have smells that are not one flower but multi-faceted, with a complex flowery character.",9
Green,"Green is a broad descriptor that refers simply to those natural smell that are green – such as the distinctive scent of cut grass, hedgerow fruits flowers, and those green notes and many green materials that help impart natural smells in a more complex accord or mix of scents.",10
Musk-Like,"These materials belong to an important fragrance note – while they are not obtained from animals, they are created to have an animal-like quality, often powdery and sometimes warm and sweet.",11
Woody,"Woody notes are party of a large odor family that includes woods such as sandalwood or cedarwood, sometimes with smoky or leather nuances. Often warm and dry notes, they impart a rich complexity that can help a fragrance last longer.",12


item,value
Blackcurrant descriptor rows,20
Cassis principal-name rows,6
Agarwood principal-name rows,5
Agarwood unique CAS,3
Cedarwood principal-name rows,19
Cedarwood unique CAS,10
Cedarwood descriptor rows,48
Musk-like primary rows,62


## 5. Allowed Official Web Sources

아래 외부 Evidence는 2026-09-01에 지정 URL을 직접 열어 확인한 짧은 요약이다. Notebook 실행 시 일반 검색이나 대체 페이지를 호출하지 않는다.


In [5]:
official_source_status_df = pd.DataFrame([
    {"source_org": "Givaudan", "source_title": "Agarwood Oil Thailand", "source_url": "https://www.givaudan.com/fragrance-beauty/fragrance-ingredients-business/natural-ingredients/agarwood-oil-thailand", "access_status": "AVAILABLE"},
    {"source_org": "Givaudan", "source_title": "Cedarwood Virginia Oil USA", "source_url": "https://www.givaudan.com/fragrance-beauty/fragrance-ingredients-business/natural-ingredients/cedarwood-virginia-oil-usa", "access_status": "AVAILABLE"},
    {"source_org": "IFF", "source_title": "Fragrance Ingredients Compendium", "source_url": "https://www.iff.com/scent/ingredients-compendium/", "access_status": "AVAILABLE"},
    {"source_org": "IFF", "source_title": "Damascone Delta", "source_url": "https://www.iff.com/scent/ingredients-compendium/damascone-delta/", "access_status": "AVAILABLE"},
    {"source_org": "IFF", "source_title": "LMR Compendium", "source_url": "https://www.iff.com/scent/lmr-compendium/", "access_status": "AVAILABLE"},
    {"source_org": "IFF", "source_title": "Edenolide", "source_url": "https://www.iff.com/scent/ingredients-compendium/edenolide/", "access_status": "AVAILABLE"},
    {"source_org": "IFF", "source_title": "Cedarwood Oil Extra", "source_url": "https://www.iff.com/scent/ingredients-compendium/cedarwood-oil-extra/", "access_status": "AVAILABLE"},
])
assert official_source_status_df["access_status"].eq("AVAILABLE").all()
show(official_source_status_df, n=10)


source_org,source_title,source_url,access_status
Givaudan,Agarwood Oil Thailand,https://www.givaudan.com/fragrance-beauty/fragrance-ingredients-business/natural-ingredients/agarwood-oil-thailand,AVAILABLE
Givaudan,Cedarwood Virginia Oil USA,https://www.givaudan.com/fragrance-beauty/fragrance-ingredients-business/natural-ingredients/cedarwood-virginia-oil-usa,AVAILABLE
IFF,Fragrance Ingredients Compendium,https://www.iff.com/scent/ingredients-compendium/,AVAILABLE
IFF,Damascone Delta,https://www.iff.com/scent/ingredients-compendium/damascone-delta/,AVAILABLE
IFF,LMR Compendium,https://www.iff.com/scent/lmr-compendium/,AVAILABLE
IFF,Edenolide,https://www.iff.com/scent/ingredients-compendium/edenolide/,AVAILABLE
IFF,Cedarwood Oil Extra,https://www.iff.com/scent/ingredients-compendium/cedarwood-oil-extra/,AVAILABLE


## 6. Evidence Table


In [6]:
IFRA_URL = "data/external/ifra/raw/ifra-fragrance-ingredient-glossary-april-2020.pdf"
IFF_COMPENDIUM = "https://www.iff.com/scent/ingredients-compendium/"

evidence_records = [
    {"term": "Woody Notes", "candidate_term": "Woody", "source_org": "IFRA", "source_title": "IFRA Fragrance Ingredient Glossary, April 2020", "source_url": IFRA_URL, "evidence_type": "PRIMARY_DESCRIPTOR_DEFINITION", "evidence_summary": "IFRA defines Woody by explicitly describing woody notes as a large odor family that includes sandalwood and cedarwood facets.", "supports_relation": "SAME_CONCEPT", "retrieved_at": RETRIEVED_AT},
    {"term": "Woody Notes", "candidate_term": "Woody", "source_org": "IFF", "source_title": "Fragrance Ingredients Compendium", "source_url": IFF_COMPENDIUM, "evidence_type": "OLFACTIVE_FAMILY", "evidence_summary": "IFF labels ingredient entries with Woody under its Olfactive Family field, confirming category-level use rather than one ingredient identity.", "supports_relation": "SAME_CONCEPT", "retrieved_at": RETRIEVED_AT},
    {"term": "Green Notes", "candidate_term": "Green", "source_org": "IFRA", "source_title": "IFRA Fragrance Ingredient Glossary, April 2020", "source_url": IFRA_URL, "evidence_type": "PRIMARY_DESCRIPTOR_DEFINITION", "evidence_summary": "IFRA calls Green a broad descriptor and uses the expression green notes within the definition.", "supports_relation": "SAME_CONCEPT", "retrieved_at": RETRIEVED_AT},
    {"term": "Green Notes", "candidate_term": "Green", "source_org": "IFF", "source_title": "Fragrance Ingredients Compendium", "source_url": IFF_COMPENDIUM, "evidence_type": "OLFACTIVE_FAMILY", "evidence_summary": "IFF uses Green as an Olfactive Family label for fragrance ingredients.", "supports_relation": "SAME_CONCEPT", "retrieved_at": RETRIEVED_AT},
    {"term": "Floral Notes", "candidate_term": "Floral", "source_org": "IFRA", "source_title": "IFRA Fragrance Ingredient Glossary, April 2020", "source_url": IFRA_URL, "evidence_type": "PRIMARY_DESCRIPTOR_DEFINITION", "evidence_summary": "IFRA states that floral notes belong to the large floral family and can represent single or multi-faceted flower character.", "supports_relation": "SAME_CONCEPT", "retrieved_at": RETRIEVED_AT},
    {"term": "Floral Notes", "candidate_term": "Floral", "source_org": "IFF", "source_title": "Fragrance Ingredients Compendium", "source_url": IFF_COMPENDIUM, "evidence_type": "OLFACTIVE_FAMILY", "evidence_summary": "IFF uses Floral as an Olfactive Family label for fragrance ingredients.", "supports_relation": "SAME_CONCEPT", "retrieved_at": RETRIEVED_AT},
    {"term": "Citruses", "candidate_term": "Citrus", "source_org": "IFRA", "source_title": "IFRA Fragrance Ingredient Glossary, April 2020", "source_url": IFRA_URL, "evidence_type": "PRIMARY_DESCRIPTOR_DEFINITION", "evidence_summary": "IFRA defines Citrus notes through the smell of fruit from the citrus family, including orange, lemon and grapefruit.", "supports_relation": "SAME_CONCEPT", "retrieved_at": RETRIEVED_AT},
    {"term": "Citruses", "candidate_term": "Citrus", "source_org": "IFF", "source_title": "Fragrance Ingredients Compendium", "source_url": IFF_COMPENDIUM, "evidence_type": "OLFACTIVE_FAMILY", "evidence_summary": "IFF uses Citrus as an Olfactive Family label; the P0 form differs only by plural labeling at the scent-category level.", "supports_relation": "SAME_CONCEPT", "retrieved_at": RETRIEVED_AT},
    {"term": "Black Currant", "candidate_term": "Blackcurrant", "source_org": "IFRA", "source_title": "IFRA Fragrance Ingredient Glossary, April 2020", "source_url": IFRA_URL, "evidence_type": "INGREDIENT_DESCRIPTOR_LINK", "evidence_summary": f"IFRA contains {int(blackcurrant_mask.sum())} Blackcurrant descriptor rows and {len(cassis_rows_df)} Cassis principal-name variants whose descriptors include Blackcurrant.", "supports_relation": "SAME_CONCEPT", "retrieved_at": RETRIEVED_AT},
    {"term": "Black Currant", "candidate_term": "Blackcurrant / Cassis", "source_org": "IFF", "source_title": "Damascone Delta", "source_url": "https://www.iff.com/scent/ingredients-compendium/damascone-delta/", "evidence_type": "OLFACTIVE_DESCRIPTION", "evidence_summary": "IFF explicitly pairs Blackcurrant with cassis in the olfactive description of Damascone Delta.", "supports_relation": "SAME_CONCEPT", "retrieved_at": RETRIEVED_AT},
    {"term": "Black Currant", "candidate_term": "Blackcurrant", "source_org": "IFF", "source_title": "LMR Compendium", "source_url": "https://www.iff.com/scent/lmr-compendium/", "evidence_type": "NATURAL_INGREDIENT_FAMILY", "evidence_summary": "IFF lists multiple Blackcurrant Bud natural materials from Ribes nigrum, supporting a concept-to-material-family distinction.", "supports_relation": "SAME_CONCEPT", "retrieved_at": RETRIEVED_AT},
    {"term": "White Musk", "candidate_term": "Musk-Like", "source_org": "IFRA", "source_title": "IFRA Fragrance Ingredient Glossary, April 2020", "source_url": IFRA_URL, "evidence_type": "PRIMARY_DESCRIPTOR_DEFINITION", "evidence_summary": f"IFRA defines Musk-Like broadly and applies it as the primary descriptor to {int(musk_like_primary_mask.sum())} diverse ingredient rows with varied secondary facets.", "supports_relation": "FAMILY", "retrieved_at": RETRIEVED_AT},
    {"term": "White Musk", "candidate_term": "Musk-Like", "source_org": "IFF", "source_title": "Edenolide", "source_url": "https://www.iff.com/scent/ingredients-compendium/edenolide/", "evidence_type": "OLFACTIVE_PROFILE", "evidence_summary": "IFF describes Edenolide as a powdery, creamy, warm white musk and says it contributes soft musk qualities, supporting White Musk as a specific musk profile rather than the whole class.", "supports_relation": "FAMILY", "retrieved_at": RETRIEVED_AT},
    {"term": "Oud", "candidate_term": "Agarwood", "source_org": "IFRA", "source_title": "IFRA Fragrance Ingredient Glossary, April 2020", "source_url": IFRA_URL, "evidence_type": "MATERIAL_FAMILY", "evidence_summary": f"IFRA lists {len(agarwood_rows_df)} Agarwood oil/extract variants across {agarwood_rows_df['cas_number'].nunique()} CAS values, all with Woody, Animal Like and Smoky descriptors.", "supports_relation": "SAME_CONCEPT", "retrieved_at": RETRIEVED_AT},
    {"term": "Oud", "candidate_term": "Agarwood", "source_org": "Givaudan", "source_title": "Agarwood Oil Thailand", "source_url": "https://www.givaudan.com/fragrance-beauty/fragrance-ingredients-business/natural-ingredients/agarwood-oil-thailand", "evidence_type": "OFFICIAL_PRODUCT_CONTEXT", "evidence_summary": "Givaudan's Agarwood Oil page explains Aquilaria resin as agarwood and uses oud for the prized perfumery concept on that same material page; this supports term-level equivalence, not identity with one oil.", "supports_relation": "SAME_CONCEPT", "retrieved_at": RETRIEVED_AT},
    {"term": "Cedar", "candidate_term": "Cedarwood", "source_org": "IFRA", "source_title": "IFRA Fragrance Ingredient Glossary, April 2020", "source_url": IFRA_URL, "evidence_type": "MATERIAL_FAMILY", "evidence_summary": f"IFRA contains {len(cedarwood_rows_df)} Cedarwood principal-name variants across {cedarwood_rows_df['cas_number'].nunique()} CAS values and {int(cedarwood_descriptor_mask.sum())} Cedarwood descriptor rows.", "supports_relation": "FAMILY", "retrieved_at": RETRIEVED_AT},
    {"term": "Cedar", "candidate_term": "Cedarwood", "source_org": "Givaudan", "source_title": "Cedarwood Virginia Oil USA", "source_url": "https://www.givaudan.com/fragrance-beauty/fragrance-ingredients-business/natural-ingredients/cedarwood-virginia-oil-usa", "evidence_type": "NATURAL_INGREDIENT_VARIANT", "evidence_summary": "Givaudan identifies Virginia cedarwood oil as Juniperus virginiana, CAS 8000-27-9, with a woody, spicy and dry profile.", "supports_relation": "FAMILY", "retrieved_at": RETRIEVED_AT},
    {"term": "Cedar", "candidate_term": "Cedarwood", "source_org": "IFF", "source_title": "Cedarwood Oil Extra", "source_url": "https://www.iff.com/scent/ingredients-compendium/cedarwood-oil-extra/", "evidence_type": "NATURAL_INGREDIENT_VARIANT", "evidence_summary": "IFF identifies Cedarwood Oil Extra as Cupressus funebris wood oil with distinct CAS values and a cedarwood, balsamic-sweet description.", "supports_relation": "FAMILY", "retrieved_at": RETRIEVED_AT},
    {"term": "Cedar", "candidate_term": "Cedarwood", "source_org": "IFF", "source_title": "LMR Compendium", "source_url": "https://www.iff.com/scent/lmr-compendium/", "evidence_type": "NATURAL_INGREDIENT_VARIANT", "evidence_summary": "IFF separately lists Cedarwood Oil Atlas from Cedrus atlantica, demonstrating another botanical cedarwood material under the woody family.", "supports_relation": "FAMILY", "retrieved_at": RETRIEVED_AT},
]

evidence_df = pd.DataFrame(evidence_records)[[
    "term", "candidate_term", "source_org", "source_title", "source_url",
    "evidence_type", "evidence_summary", "supports_relation", "retrieved_at",
]]
evidence_df.to_csv(EVIDENCE_PATH, index=False, encoding="utf-8-sig")
assert set(evidence_df["term"]) == set(P0_TERMS)
assert not evidence_df[["source_org", "source_title", "source_url", "evidence_summary"]].eq("").any().any()
show(evidence_df, n=25)


term,candidate_term,source_org,source_title,source_url,evidence_type,evidence_summary,supports_relation,retrieved_at
Woody Notes,Woody,IFRA,"IFRA Fragrance Ingredient Glossary, April 2020",data/external/ifra/raw/ifra-fragrance-ingredient-glossary-april-2020.pdf,PRIMARY_DESCRIPTOR_DEFINITION,IFRA defines Woody by explicitly describing woody notes as a large odor family that includes sandalwood and cedarwood facets.,SAME_CONCEPT,2026-09-01
Woody Notes,Woody,IFF,Fragrance Ingredients Compendium,https://www.iff.com/scent/ingredients-compendium/,OLFACTIVE_FAMILY,"IFF labels ingredient entries with Woody under its Olfactive Family field, confirming category-level use rather than one ingredient identity.",SAME_CONCEPT,2026-09-01
Green Notes,Green,IFRA,"IFRA Fragrance Ingredient Glossary, April 2020",data/external/ifra/raw/ifra-fragrance-ingredient-glossary-april-2020.pdf,PRIMARY_DESCRIPTOR_DEFINITION,IFRA calls Green a broad descriptor and uses the expression green notes within the definition.,SAME_CONCEPT,2026-09-01
Green Notes,Green,IFF,Fragrance Ingredients Compendium,https://www.iff.com/scent/ingredients-compendium/,OLFACTIVE_FAMILY,IFF uses Green as an Olfactive Family label for fragrance ingredients.,SAME_CONCEPT,2026-09-01
Floral Notes,Floral,IFRA,"IFRA Fragrance Ingredient Glossary, April 2020",data/external/ifra/raw/ifra-fragrance-ingredient-glossary-april-2020.pdf,PRIMARY_DESCRIPTOR_DEFINITION,IFRA states that floral notes belong to the large floral family and can represent single or multi-faceted flower character.,SAME_CONCEPT,2026-09-01
Floral Notes,Floral,IFF,Fragrance Ingredients Compendium,https://www.iff.com/scent/ingredients-compendium/,OLFACTIVE_FAMILY,IFF uses Floral as an Olfactive Family label for fragrance ingredients.,SAME_CONCEPT,2026-09-01
Citruses,Citrus,IFRA,"IFRA Fragrance Ingredient Glossary, April 2020",data/external/ifra/raw/ifra-fragrance-ingredient-glossary-april-2020.pdf,PRIMARY_DESCRIPTOR_DEFINITION,"IFRA defines Citrus notes through the smell of fruit from the citrus family, including orange, lemon and grapefruit.",SAME_CONCEPT,2026-09-01
Citruses,Citrus,IFF,Fragrance Ingredients Compendium,https://www.iff.com/scent/ingredients-compendium/,OLFACTIVE_FAMILY,IFF uses Citrus as an Olfactive Family label; the P0 form differs only by plural labeling at the scent-category level.,SAME_CONCEPT,2026-09-01
Black Currant,Blackcurrant,IFRA,"IFRA Fragrance Ingredient Glossary, April 2020",data/external/ifra/raw/ifra-fragrance-ingredient-glossary-april-2020.pdf,INGREDIENT_DESCRIPTOR_LINK,IFRA contains 20 Blackcurrant descriptor rows and 6 Cassis principal-name variants whose descriptors include Blackcurrant.,SAME_CONCEPT,2026-09-01
Black Currant,Blackcurrant / Cassis,IFF,Damascone Delta,https://www.iff.com/scent/ingredients-compendium/damascone-delta/,OLFACTIVE_DESCRIPTION,IFF explicitly pairs Blackcurrant with cassis in the olfactive description of Damascone Delta.,SAME_CONCEPT,2026-09-01


## 7. Decision Criteria


In [7]:
decision_criteria_df = pd.DataFrame([
    {"relation": "SAME_CONCEPT", "criterion": "Official source directly or very clearly supports the same scent concept; not merely similar and not one ingredient identity."},
    {"relation": "FAMILY", "criterion": "One term is a broader/narrower scent profile or material family; multiple variants exist and full synonymy is unsafe."},
    {"relation": "RELATED", "criterion": "Official olfactive relation exists, but neither equivalence nor hierarchy is supported."},
    {"relation": "UNRESOLVED", "criterion": "Allowed official evidence is insufficient or concept/ingredient separation remains ambiguous."},
])
show(decision_criteria_df)


relation,criterion
SAME_CONCEPT,Official source directly or very clearly supports the same scent concept; not merely similar and not one ingredient identity.
FAMILY,One term is a broader/narrower scent profile or material family; multiple variants exist and full synonymy is unsafe.
RELATED,"Official olfactive relation exists, but neither equivalence nor hierarchy is supported."
UNRESOLVED,Allowed official evidence is insufficient or concept/ingredient separation remains ambiguous.


## 8. P0 Decisions


In [8]:
decision_specs = [
    {"term": "Woody Notes", "candidate": "Woody", "final_relation": "SAME_CONCEPT", "canonical_term": "Woody", "confidence": "HIGH", "reason": "IFRA itself defines Woody using the phrase woody notes, and IFF uses Woody as an olfactive family.", "manual_review_required": False},
    {"term": "Green Notes", "candidate": "Green", "final_relation": "SAME_CONCEPT", "canonical_term": "Green", "confidence": "HIGH", "reason": "IFRA defines Green as a broad descriptor and explicitly refers to green notes.", "manual_review_required": False},
    {"term": "Floral Notes", "candidate": "Floral", "final_relation": "SAME_CONCEPT", "canonical_term": "Floral", "confidence": "HIGH", "reason": "IFRA explicitly defines floral notes as belonging to the floral family; IFF uses Floral as an olfactive family.", "manual_review_required": False},
    {"term": "Citruses", "candidate": "Citrus", "final_relation": "SAME_CONCEPT", "canonical_term": "Citrus", "confidence": "HIGH", "reason": "The P0 plural label and IFRA's Citrus-notes category refer to the same citrus-family scent category.", "manual_review_required": False},
    {"term": "Black Currant", "candidate": "Blackcurrant", "final_relation": "SAME_CONCEPT", "canonical_term": "Blackcurrant", "confidence": "HIGH", "reason": "This is an orthographic canonicalization; IFF explicitly pairs Blackcurrant with cassis, and IFRA links Cassis materials to Blackcurrant descriptors.", "manual_review_required": False},
    {"term": "White Musk", "candidate": "Musk-Like", "final_relation": "FAMILY", "canonical_term": "White Musk", "confidence": "MEDIUM", "reason": "IFF supports white musk as a specific soft/powdery musk profile, while IFRA Musk-Like covers a wider set of materials and facets.", "manual_review_required": True},
    {"term": "Oud", "candidate": "Agarwood", "final_relation": "SAME_CONCEPT", "canonical_term": "Agarwood", "confidence": "HIGH", "reason": "Givaudan uses oud in the official Agarwood Oil material context; IFRA shows that Agarwood maps to multiple ingredient variants, so only term-level equivalence is confirmed.", "manual_review_required": False},
    {"term": "Cedar", "candidate": "Cedarwood", "final_relation": "FAMILY", "canonical_term": "Cedar", "confidence": "MEDIUM", "reason": "Official sources document cedarwood materials from Juniperus, Cupressus and Cedrus with different CAS/species; collapsing Cedar to one ingredient or full synonym is unsafe.", "manual_review_required": True},
]

decisions_df = pd.DataFrame(decision_specs)
decisions_df = decisions_df.merge(p0_stage17_df[["term", "stage17_status"]], on="term", how="left")
evidence_counts = evidence_df.groupby("term").size().rename("evidence_count")
decisions_df = decisions_df.merge(evidence_counts, on="term", how="left")
decisions_df = decisions_df[[
    "term", "candidate", "stage17_status", "final_relation", "canonical_term",
    "confidence", "reason", "evidence_count", "manual_review_required",
]]
decisions_df.to_csv(DECISION_PATH, index=False, encoding="utf-8-sig")

assert len(decisions_df) == 8 and decisions_df["evidence_count"].ge(2).all()
assert decisions_df.loc[decisions_df["confidence"].ne("HIGH"), "manual_review_required"].all()
show(decisions_df, n=8)


term,candidate,stage17_status,final_relation,canonical_term,confidence,reason,evidence_count,manual_review_required
Woody Notes,Woody,UNMATCHED,SAME_CONCEPT,Woody,HIGH,"IFRA itself defines Woody using the phrase woody notes, and IFF uses Woody as an olfactive family.",2,False
Green Notes,Green,UNMATCHED,SAME_CONCEPT,Green,HIGH,IFRA defines Green as a broad descriptor and explicitly refers to green notes.,2,False
Floral Notes,Floral,UNMATCHED,SAME_CONCEPT,Floral,HIGH,IFRA explicitly defines floral notes as belonging to the floral family; IFF uses Floral as an olfactive family.,2,False
Citruses,Citrus,UNMATCHED,SAME_CONCEPT,Citrus,HIGH,The P0 plural label and IFRA's Citrus-notes category refer to the same citrus-family scent category.,2,False
Black Currant,Blackcurrant,UNMATCHED,SAME_CONCEPT,Blackcurrant,HIGH,"This is an orthographic canonicalization; IFF explicitly pairs Blackcurrant with cassis, and IFRA links Cassis materials to Blackcurrant descriptors.",3,False
White Musk,Musk-Like,UNMATCHED,FAMILY,White Musk,MEDIUM,"IFF supports white musk as a specific soft/powdery musk profile, while IFRA Musk-Like covers a wider set of materials and facets.",2,True
Oud,Agarwood,CANDIDATE_ONLY,SAME_CONCEPT,Agarwood,HIGH,"Givaudan uses oud in the official Agarwood Oil material context; IFRA shows that Agarwood maps to multiple ingredient variants, so only term-level equivalence is confirmed.",2,False
Cedar,Cedarwood,CANDIDATE_ONLY,FAMILY,Cedar,MEDIUM,"Official sources document cedarwood materials from Juniperus, Cupressus and Cedrus with different CAS/species; collapsing Cedar to one ingredient or full synonym is unsafe.",4,True


## 9. Canonicalization vs Ingredient Mapping


In [9]:
mapping_separation_df = pd.DataFrame([
    {"term": "Oud", "canonicalization": "Oud -> SAME_CONCEPT -> Agarwood", "ingredient_mapping": "Agarwood -> material family -> multiple IFRA agarwood oil/extract rows", "prohibited_interpretation": "Oud Note = Agarwood Oil Thailand"},
    {"term": "Black Currant", "canonicalization": "Black Currant -> SAME_CONCEPT -> Blackcurrant", "ingredient_mapping": "Blackcurrant concept -> includes Cassis bud materials and other ingredient descriptors", "prohibited_interpretation": "Black Currant Note = one Cassis bud absolute"},
    {"term": "Cedar", "canonicalization": "Cedar remains a broader concept", "ingredient_mapping": "Cedar -> FAMILY -> multiple Cedarwood materials/species", "prohibited_interpretation": "Cedar Note = one Cedarwood oil"},
    {"term": "White Musk", "canonicalization": "White Musk remains a specific profile", "ingredient_mapping": "White Musk -> FAMILY -> broader Musk-Like class", "prohibited_interpretation": "White Musk = every Musk-Like ingredient"},
])
show(mapping_separation_df)


term,canonicalization,ingredient_mapping,prohibited_interpretation
Oud,Oud -> SAME_CONCEPT -> Agarwood,Agarwood -> material family -> multiple IFRA agarwood oil/extract rows,Oud Note = Agarwood Oil Thailand
Black Currant,Black Currant -> SAME_CONCEPT -> Blackcurrant,Blackcurrant concept -> includes Cassis bud materials and other ingredient descriptors,Black Currant Note = one Cassis bud absolute
Cedar,Cedar remains a broader concept,Cedar -> FAMILY -> multiple Cedarwood materials/species,Cedar Note = one Cedarwood oil
White Musk,White Musk remains a specific profile,White Musk -> FAMILY -> broader Musk-Like class,White Musk = every Musk-Like ingredient


## 10. Build Scent Term Dictionary v0.1


In [10]:
dictionary_specs = {
    "Woody Notes": {"parent_term": "", "iframapped_term": "Woody", "decision": "CONFIRMED_CANONICALIZATION", "notes": "Category-label wording only; no ingredient identity implied."},
    "Green Notes": {"parent_term": "", "iframapped_term": "Green", "decision": "CONFIRMED_CANONICALIZATION", "notes": "Category-label wording only; no ingredient identity implied."},
    "Floral Notes": {"parent_term": "", "iframapped_term": "Floral", "decision": "CONFIRMED_CANONICALIZATION", "notes": "Category-label wording only; no ingredient identity implied."},
    "Citruses": {"parent_term": "", "iframapped_term": "Citrus", "decision": "CONFIRMED_CANONICALIZATION", "notes": "Plural category label canonicalized to IFRA Citrus."},
    "Black Currant": {"parent_term": "", "iframapped_term": "Blackcurrant", "decision": "CONFIRMED_CANONICALIZATION", "notes": "Blackcurrant/Cassis evidence does not identify the Note with one natural material."},
    "White Musk": {"parent_term": "Musk-Like", "iframapped_term": "Musk-Like", "decision": "KEEP_SEPARATE_FAMILY_LINK", "notes": "Specific soft/powdery musk profile; manual review retained."},
    "Oud": {"parent_term": "", "iframapped_term": "Agarwood principal-name family", "decision": "CONFIRMED_CANONICALIZATION", "notes": "Canonical term only; not mapped to one oil/extract variant."},
    "Cedar": {"parent_term": "Woody", "iframapped_term": "Cedarwood", "decision": "KEEP_SEPARATE_FAMILY_LINK", "notes": "Broad note linked to multiple cedarwood materials/species; manual review retained."},
}

dictionary_rows = []
for row in decisions_df.itertuples(index=False):
    spec = dictionary_specs[row.term]
    dictionary_rows.append({
        "term": row.term,
        "term_type": "NOTE",
        "canonical_term": row.canonical_term,
        "relation_type": row.final_relation,
        "parent_term": spec["parent_term"],
        "iframapped_term": spec["iframapped_term"],
        "decision": spec["decision"],
        "confidence": row.confidence,
        "evidence_count": row.evidence_count,
        "notes": spec["notes"],
        "version": VERSION,
    })

dictionary_df = pd.DataFrame(dictionary_rows)[[
    "term", "term_type", "canonical_term", "relation_type", "parent_term",
    "iframapped_term", "decision", "confidence", "evidence_count", "notes", "version",
]]
dictionary_df.to_csv(DICTIONARY_PATH, index=False, encoding="utf-8-sig")
assert len(dictionary_df) == 8 and dictionary_df["version"].eq(VERSION).all()
show(dictionary_df, n=8)


term,term_type,canonical_term,relation_type,parent_term,iframapped_term,decision,confidence,evidence_count,notes,version
Woody Notes,NOTE,Woody,SAME_CONCEPT,,Woody,CONFIRMED_CANONICALIZATION,HIGH,2,Category-label wording only; no ingredient identity implied.,v0.1
Green Notes,NOTE,Green,SAME_CONCEPT,,Green,CONFIRMED_CANONICALIZATION,HIGH,2,Category-label wording only; no ingredient identity implied.,v0.1
Floral Notes,NOTE,Floral,SAME_CONCEPT,,Floral,CONFIRMED_CANONICALIZATION,HIGH,2,Category-label wording only; no ingredient identity implied.,v0.1
Citruses,NOTE,Citrus,SAME_CONCEPT,,Citrus,CONFIRMED_CANONICALIZATION,HIGH,2,Plural category label canonicalized to IFRA Citrus.,v0.1
Black Currant,NOTE,Blackcurrant,SAME_CONCEPT,,Blackcurrant,CONFIRMED_CANONICALIZATION,HIGH,3,Blackcurrant/Cassis evidence does not identify the Note with one natural material.,v0.1
White Musk,NOTE,White Musk,FAMILY,Musk-Like,Musk-Like,KEEP_SEPARATE_FAMILY_LINK,MEDIUM,2,Specific soft/powdery musk profile; manual review retained.,v0.1
Oud,NOTE,Agarwood,SAME_CONCEPT,,Agarwood principal-name family,CONFIRMED_CANONICALIZATION,HIGH,2,Canonical term only; not mapped to one oil/extract variant.,v0.1
Cedar,NOTE,Cedar,FAMILY,Woody,Cedarwood,KEEP_SEPARATE_FAMILY_LINK,MEDIUM,4,Broad note linked to multiple cedarwood materials/species; manual review retained.,v0.1


## 11. Recalculate Coverage with Confirmed SAME_CONCEPT Only


In [11]:
confirmed_same_terms = set(
    decisions_df.loc[
        decisions_df["final_relation"].eq("SAME_CONCEPT")
        & decisions_df["confidence"].eq("HIGH")
        & ~decisions_df["manual_review_required"],
        "term",
    ]
)
assert confirmed_same_terms == {"Woody Notes", "Green Notes", "Floral Notes", "Citruses", "Black Currant", "Oud"}

stage17_strict_mask = note_match_df["final_auto_status"].eq("STRICT_MATCH")
stage18_strict_mask = stage17_strict_mask | note_match_df["note"].isin(confirmed_same_terms)

total_notes = len(note_match_df)
total_note_occurrences = int(note_match_df["occurrence_count"].sum())
stage17_strict_count = int(stage17_strict_mask.sum())
stage18_strict_count = int(stage18_strict_mask.sum())
stage17_vocab_coverage = ratio(stage17_strict_count, total_notes)
stage18_vocab_coverage = ratio(stage18_strict_count, total_notes)
stage17_strict_occurrences = int(note_match_df.loc[stage17_strict_mask, "occurrence_count"].sum())
stage18_strict_occurrences = int(note_match_df.loc[stage18_strict_mask, "occurrence_count"].sum())
stage17_occurrence_coverage = ratio(stage17_strict_occurrences, total_note_occurrences)
stage18_occurrence_coverage = ratio(stage18_strict_occurrences, total_note_occurrences)

stage17_strict_terms = set(note_match_df.loc[stage17_strict_mask, "note"])
stage18_strict_terms = stage17_strict_terms | confirmed_same_terms
stage17_perfume_hits = 0
stage18_perfume_hits = 0
total_perfumes = 0

with JSONL_PATH.open("r", encoding="utf-8") as stream:
    for line in stream:
        record = json.loads(line)
        total_perfumes += 1
        notes = record.get("notes", {}) or {}
        tiered = notes.get("tiered", {}) or {}
        perfume_note_names = []
        for tier in ("top", "middle", "base"):
            perfume_note_names.extend(item.get("name", "") for item in (tiered.get(tier, []) or []))
        perfume_note_names.extend(item.get("name", "") for item in (notes.get("flat", []) or []))
        note_set = set(perfume_note_names)
        stage17_perfume_hits += bool(note_set & stage17_strict_terms)
        stage18_perfume_hits += bool(note_set & stage18_strict_terms)

stage17_perfume_coverage = ratio(stage17_perfume_hits, total_perfumes)
stage18_perfume_coverage = ratio(stage18_perfume_hits, total_perfumes)

assert stage17_strict_count == int(stage17_metrics["strict_matched_notes"])
assert abs(stage17_vocab_coverage - stage17_metrics["strict_note_vocabulary_coverage"]) < 1e-12
assert abs(stage17_occurrence_coverage - stage17_metrics["strict_note_occurrence_weighted_coverage"]) < 1e-12
assert stage17_perfume_hits == int(stage17_metrics["perfumes_with_at_least_one_strict_matched_note"])

coverage_comparison_df = pd.DataFrame([
    {"metric": "Strict matched note count", "stage17": stage17_strict_count, "stage18": stage18_strict_count, "delta": stage18_strict_count - stage17_strict_count},
    {"metric": "Vocabulary coverage", "stage17": stage17_vocab_coverage, "stage18": stage18_vocab_coverage, "delta": stage18_vocab_coverage - stage17_vocab_coverage},
    {"metric": "Occurrence-weighted coverage", "stage17": stage17_occurrence_coverage, "stage18": stage18_occurrence_coverage, "delta": stage18_occurrence_coverage - stage17_occurrence_coverage},
    {"metric": "Perfume-level coverage", "stage17": stage17_perfume_coverage, "stage18": stage18_perfume_coverage, "delta": stage18_perfume_coverage - stage17_perfume_coverage},
])
show(coverage_comparison_df)


metric,stage17,stage18,delta
Strict matched note count,159.000000,165.000000,6.000000
Vocabulary coverage,0.063020,0.065398,0.002378
Occurrence-weighted coverage,0.497042,0.526073,0.029031
Perfume-level coverage,0.923111,0.942720,0.019609


## 12. Term-Level Coverage Contribution


In [12]:
coverage_contribution_df = p0_stage17_df.merge(
    decisions_df[["term", "final_relation", "confidence", "manual_review_required"]],
    on="term",
    how="left",
)
coverage_contribution_df["applied_to_stage18_strict"] = coverage_contribution_df["term"].isin(confirmed_same_terms)
coverage_contribution_df["added_occurrences"] = np.where(
    coverage_contribution_df["applied_to_stage18_strict"], coverage_contribution_df["occurrence_count"], 0
)
show(coverage_contribution_df, n=8)


term,perfume_count,occurrence_count,stage17_status,stage17_candidate,final_relation,confidence,manual_review_required,applied_to_stage18_strict,added_occurrences
Woody Notes,8278,8385,UNMATCHED,,SAME_CONCEPT,HIGH,False,True,8385
Green Notes,4529,4538,UNMATCHED,,SAME_CONCEPT,HIGH,False,True,4538
Floral Notes,4436,4505,UNMATCHED,,SAME_CONCEPT,HIGH,False,True,4505
Citruses,4982,4993,UNMATCHED,,SAME_CONCEPT,HIGH,False,True,4993
Black Currant,5560,5566,UNMATCHED,,SAME_CONCEPT,HIGH,False,True,5566
White Musk,8339,8371,UNMATCHED,,FAMILY,MEDIUM,True,False,0
Oud,3780,3895,CANDIDATE_ONLY,Agarwood,SAME_CONCEPT,HIGH,False,True,3895
Cedar,22305,22502,CANDIDATE_ONLY,"Cedar leaf oil | Cedar leaf oil, China | Cedar leaf oil, East Canada | Cedar leaf oil, rectified | Cedarwood",FAMILY,MEDIUM,True,False,0


## 13. Write Summary


In [13]:
relation_counts = decisions_df["final_relation"].value_counts().reindex(
    ["SAME_CONCEPT", "FAMILY", "RELATED", "UNRESOLVED"], fill_value=0
)

term_decision_lines = "\n".join(
    f"- **{row.term} -> {row.candidate}**: `{row.final_relation}` / `{row.confidence}` - {row.reason}"
    for row in decisions_df.itertuples(index=False)
)
same_lines = "\n".join(
    f"- {row.term} -> {row.canonical_term}"
    for row in decisions_df[decisions_df["final_relation"].eq("SAME_CONCEPT")].itertuples(index=False)
)
family_lines = "\n".join(
    f"- {row.term} -> {row.candidate}: {row.reason}"
    for row in decisions_df[decisions_df["final_relation"].eq("FAMILY")].itertuples(index=False)
)

summary_text = f"""# Scent Term Dictionary Validation

## 결과 요약

### 결론

P0 8개 공식자료 검증 결과:

- SAME_CONCEPT: {relation_counts['SAME_CONCEPT']}개
- FAMILY: {relation_counts['FAMILY']}개
- RELATED: {relation_counts['RELATED']}개
- UNRESOLVED: {relation_counts['UNRESOLVED']}개

확정된 HIGH-confidence SAME_CONCEPT 6개만 Stage 18 strict mapping에 추가했다. FAMILY인 White Musk와 Cedar는 Coverage에 포함하지 않았다.

## 상세

### 1. 분석 목적

17번의 고빈도 unmatched / alias 후보 중 P0 8개를 IFRA, Givaudan, IFF 공식 자료로만 검증하고 canonical scent term과 ingredient family를 분리한 사전 v0.1을 구축했다.

### 2. 검증 대상

{', '.join(P0_TERMS)}

### 3. 사용한 공식 Source

- IFRA Fragrance Ingredient Glossary, April 2020: `{IFRA_URL}`
- Givaudan Agarwood Oil Thailand: https://www.givaudan.com/fragrance-beauty/fragrance-ingredients-business/natural-ingredients/agarwood-oil-thailand
- Givaudan Cedarwood Virginia Oil USA: https://www.givaudan.com/fragrance-beauty/fragrance-ingredients-business/natural-ingredients/cedarwood-virginia-oil-usa
- IFF Fragrance Ingredients Compendium: https://www.iff.com/scent/ingredients-compendium/
- IFF Damascone Delta: https://www.iff.com/scent/ingredients-compendium/damascone-delta/
- IFF LMR Compendium: https://www.iff.com/scent/lmr-compendium/
- IFF Edenolide: https://www.iff.com/scent/ingredients-compendium/edenolide/
- IFF Cedarwood Oil Extra: https://www.iff.com/scent/ingredients-compendium/cedarwood-oil-extra/

지정 URL 7개는 {RETRIEVED_AT}에 모두 접근 가능했다. 검색 snippet이나 대체 출처는 사용하지 않았다.

### 4. 용어별 판정 결과

{term_decision_lines}

### 5. SAME_CONCEPT 결과

{same_lines}

이는 향 용어 canonicalization이며 특정 ingredient 하나와의 동일성을 뜻하지 않는다.

### 6. FAMILY 결과

{family_lines}

White Musk는 IFRA Musk-Like 전체보다 구체적인 profile로, Cedar는 여러 식물 종과 CAS의 Cedarwood material을 포괄하는 상위 note concept로 유지했다.

### 7. RELATED / UNRESOLVED 결과

이번 P0에서는 RELATED와 UNRESOLVED가 없었다. 이는 모든 후보를 강제로 연결한 결과가 아니라, 6개는 공식 정의/직접 표현이 충분했고 나머지 2개는 SAME_CONCEPT로 올리지 않고 FAMILY로 보존한 결과다.

### 8. Coverage 변화

- Strict matched Note: {stage17_strict_count:,} -> {stage18_strict_count:,} (+{stage18_strict_count - stage17_strict_count:,})
- Vocabulary coverage: {stage17_vocab_coverage:.2%} -> {stage18_vocab_coverage:.2%} ({(stage18_vocab_coverage - stage17_vocab_coverage) * 100:+.2f} percentage points)
- Occurrence-weighted coverage: {stage17_occurrence_coverage:.2%} -> {stage18_occurrence_coverage:.2%} ({(stage18_occurrence_coverage - stage17_occurrence_coverage) * 100:+.2f} percentage points)
- Perfume-level coverage: {stage17_perfume_coverage:.2%} -> {stage18_perfume_coverage:.2%} ({(stage18_perfume_coverage - stage17_perfume_coverage) * 100:+.2f} percentage points)

Coverage는 판정 이후에 계산했으며 FAMILY / RELATED는 strict에 포함하지 않았다.

### 9. 향 용어 사전 구조

`scent_term_dictionary_v0.1.csv`는 term, canonical term, relation, parent, IFRA mapping, confidence와 version을 보존한다. `scent_term_evidence_v0.1.csv`는 각 판정을 공식 Source URL과 Evidence 요약으로 추적한다.

### 10. 이번 분석에서 검증된 것

- category label의 표현 차이와 canonical form
- Blackcurrant / Cassis의 공식 향 표현 관계
- Oud / Agarwood의 향수 용어 관계와 복수 ingredient family의 분리
- White Musk와 Cedar를 broader class/material family와 구분해야 한다는 점

### 11. 아직 검증되지 않은 것

- P0 이외 unmatched Note
- 각 canonical scent concept를 모든 실제 ingredient에 연결하는 완전한 ontology
- 향 강도, 배합 비율, 추천 ranking score
- 자연어 semantic bridge

### 12. 다음 단계

1. MEDIUM-confidence FAMILY 두 건을 향료 전문가가 검토한다.
2. 다음 고빈도 unmatched 묶음을 같은 공식 Source 제한으로 확장한다.
3. 검증된 SAME_CONCEPT만 versioned dictionary에 추가하고 Coverage와 추천 평가는 분리한다.
"""
SUMMARY_PATH.write_text(summary_text, encoding="utf-8")
print(f"Saved summary -> {SUMMARY_PATH.relative_to(PROJECT_ROOT)}")


Saved summary -> analysis_outputs\18_scent_term_dictionary_validation_summary.md


## 14. Artifact Validation


In [14]:
expected_outputs = [DICTIONARY_PATH, EVIDENCE_PATH, DECISION_PATH, SUMMARY_PATH]
assert all(path.exists() and path.stat().st_size > 0 for path in expected_outputs)

dictionary_check = pd.read_csv(DICTIONARY_PATH, keep_default_na=False)
evidence_check = pd.read_csv(EVIDENCE_PATH, keep_default_na=False)
decision_check = pd.read_csv(DECISION_PATH, keep_default_na=False)

assert dictionary_check.columns.tolist() == [
    "term", "term_type", "canonical_term", "relation_type", "parent_term",
    "iframapped_term", "decision", "confidence", "evidence_count", "notes", "version",
]
assert evidence_check.columns.tolist() == [
    "term", "candidate_term", "source_org", "source_title", "source_url",
    "evidence_type", "evidence_summary", "supports_relation", "retrieved_at",
]
assert decision_check.columns.tolist() == [
    "term", "candidate", "stage17_status", "final_relation", "canonical_term",
    "confidence", "reason", "evidence_count", "manual_review_required",
]
assert len(dictionary_check) == len(decision_check) == 8
assert len(evidence_check) == len(evidence_df)
assert set(dictionary_check["term"]) == set(P0_TERMS)
assert dictionary_check["evidence_count"].sum() == len(evidence_check)
assert decision_check.loc[decision_check["confidence"].ne("HIGH"), "manual_review_required"].all()
assert not decision_check.loc[decision_check["final_relation"].eq("FAMILY"), "term"].isin(confirmed_same_terms).any()
print(f"Validated {len(expected_outputs)} artifacts, 8 terms, {len(evidence_check)} evidence rows.")


Validated 4 artifacts, 8 terms, 19 evidence rows.


## 15. Final Summary


In [15]:
print("[P0 Validation]")
print("Total Terms: 8")
for relation in ["SAME_CONCEPT", "FAMILY", "RELATED", "UNRESOLVED"]:
    print(f"{relation}: {int(relation_counts[relation])}")

print("\n[Coverage]")
print(f"Stage17 Vocabulary Coverage: {stage17_vocab_coverage:.2%}")
print(f"Stage18 Vocabulary Coverage: {stage18_vocab_coverage:.2%}")
print(f"Stage17 Occurrence-weighted Coverage: {stage17_occurrence_coverage:.2%}")
print(f"Stage18 Occurrence-weighted Coverage: {stage18_occurrence_coverage:.2%}")
print(f"Stage17 Perfume-level Coverage: {stage17_perfume_coverage:.2%}")
print(f"Stage18 Perfume-level Coverage: {stage18_perfume_coverage:.2%}")

print("\n[Artifacts]")
for path in expected_outputs:
    print(path.relative_to(PROJECT_ROOT))


[P0 Validation]
Total Terms: 8
SAME_CONCEPT: 6
FAMILY: 2
RELATED: 0
UNRESOLVED: 0

[Coverage]
Stage17 Vocabulary Coverage: 6.30%
Stage18 Vocabulary Coverage: 6.54%
Stage17 Occurrence-weighted Coverage: 49.70%
Stage18 Occurrence-weighted Coverage: 52.61%
Stage17 Perfume-level Coverage: 92.31%
Stage18 Perfume-level Coverage: 94.27%

[Artifacts]
data\scent_knowledge\scent_term_dictionary_v0.1.csv
data\scent_knowledge\scent_term_evidence_v0.1.csv
analysis_outputs\18_scent_term_validation_decisions.csv
analysis_outputs\18_scent_term_dictionary_validation_summary.md
